In [1]:
import os
import re
import sys
import pandas as pd
import shutil
from openpyxl import load_workbook
from openpyxl.cell.cell import MergedCell
from openpyxl.utils import get_column_letter
from configparser import ConfigParser
from pathlib import Path
from datetime import datetime

ini_path = Path("config.ini").resolve()

### Base Directory Definitions

In [2]:
def load_paths(ini_path: str, section: str = "paths"):
    s = ConfigParser()
    s.read(ini_path)
    s = s[section]

    base = Path(s.get("base_dir", ".")).expanduser().resolve()

    template = base / s.get("template_dir", "templates") / s["template_file"]
    input_dir = base / s.get("input_dir", "input_files")
    output_dir = base / s.get("output_dir", "output")
    output_dir.mkdir(parents=True, exist_ok=True)

    input_file = s.get("input_file", "").strip()
    input_path = (input_dir / input_file) if input_file else sorted(input_dir.glob("*.xlsx"))[0]

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = output_dir / f'{s.get("output_prefix","output_")}{stamp}{s.get("output_ext",".xlsx")}'

    print("Input directory:", input_dir)

    return template, input_path, output_path
template_path, input_path, output_path = load_paths("PHM01_Report.ini")

Input directory: /Users/reinaldoburgos/sources/My_Space/my_projects/PHM01_Test/input_files


In [3]:
shutil.copy(template_path,output_path)

PosixPath('/Users/reinaldoburgos/sources/My_Space/my_projects/PHM01_Test/output_files/PHM01_Report_filled_20251231_091725.xlsx')

In [4]:
print("Template:", template_path)
print("Input:", input_path)
print("Output:", output_path)

Template: /Users/reinaldoburgos/sources/My_Space/my_projects/PHM01_Test/templates/PHM01_template.xlsx
Input: /Users/reinaldoburgos/sources/My_Space/my_projects/PHM01_Test/input_files/PHM01_Input_Data.xlsx
Output: /Users/reinaldoburgos/sources/My_Space/my_projects/PHM01_Test/output_files/PHM01_Report_filled_20251231_091725.xlsx


### TAB 2

In [5]:
df_tab2 = pd.read_excel(input_path, sheet_name="Health Assessment(HRA)", skiprows=8, usecols=('Region','Metric','Seq_Memb_Id','Month'))
df_tab2.columns = df_tab2.columns.str.strip()

print(df_tab2.columns)
print(df_tab2.head(15))



# -------------------------
# 2) UTIL: NORMALIZE COLUMNS
# -------------------------
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip().str.lower().str.replace(" ", "_")
    return df


Index(['Region', 'Metric', 'Seq_Memb_Id', 'Month'], dtype='object')
      Region                    Metric  Seq_Memb_Id Month
0   Region 8  Initial HRA Completion #     63564672   May
1   Region 5   Initial HRA Enrollees #     62950707   Aug
2   Region 3   Initial HRA Enrollees #      7678280   Jan
3   Region 1    Annual HRA Completed #     75792506   Jun
4   Region 4  Initial HRA Completion #     25405295   Jun
5   Region 6  Initial HRA Completion #     11152885   Dec
6   Region 3    Annual HRA Completed #     39847612   Nov
7   Region 1    Annual HRA Enrollees #      5289692   Apr
8   Region 6    Annual HRA Enrollees #     30942098   Jun
9   Region 2   Initial HRA Enrollees #     26565423   Dec
10  Region 5    Annual HRA Enrollees #     16736731   Jan
11  Region 2    Annual HRA Completed #     90098800   Aug
12  Region 7  Initial HRA Completion #     57637595   Jun
13  Region 1    Annual HRA Enrollees #     56228329   Jan
14  Region 3  Initial HRA Completion #     61299999   Mar


In [6]:
sheet_name = "Health Assessment(HRA)"

# -----------------------
# CONSTANTS
# -----------------------
REGIONS = [f"Region {i}" for i in range(1, 9)]
MONTH_ORDER = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# Month data columns are fixed to C..N
MONTH_START_COL = 3

# Your metrics from counts
METRICS_TO_WRITE = [
    "Initial HRA Enrollees #",
    "Initial HRA Completion #",
    "Annual HRA Enrollees #",
    "Annual HRA Completed #",
]

# -----------------------
# HELPERS
# -----------------------
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip().str.lower().str.replace(" ", "_")
    return df

def detect_header_row(excel_path, sheet_name, required=("region","metric","seq_memb_id","month"), max_rows=80):
    preview = pd.read_excel(excel_path, sheet_name=sheet_name, header=None, nrows=max_rows, dtype=str)
    req = set([r.lower() for r in required])
    for i in range(max_rows):
        row_vals = preview.iloc[i].dropna().astype(str).str.strip().str.lower().tolist()
        if req.issubset(set(row_vals)):
            return i
    return None

def month_to_col_fixed(month_str: str):
    m = str(month_str).strip()
    if m not in MONTH_ORDER:
        return None
    return MONTH_START_COL + MONTH_ORDER.index(m)

def _norm(x): return str(x).strip().lower()

def find_first_match_anywhere(ws, text, max_rows=2000, max_cols=80):
    tgt = _norm(text)
    for r in range(1, min(ws.max_row, max_rows) + 1):
        for c in range(1, min(ws.max_column, max_cols) + 1):
            v = ws.cell(r, c).value
            if v is not None and _norm(v) == tgt:
                return (r, c)
    return None

def find_row_in_column(ws, col_idx, text, start_row=1, end_row=None):
    tgt = _norm(text)
    end_row = end_row or ws.max_row
    for r in range(start_row, end_row + 1):
        v = ws.cell(r, col_idx).value
        if v is not None and _norm(v) == tgt:
            return r
    return None

def fill_tab2_fixed_month_columns(template_path, output_path, counts_df, sheet_name):
    wb = load_workbook(template_path)
    ws = wb[sheet_name]

    # detect label column by locating "Region 1"
    r1 = find_first_match_anywhere(ws, "Region 1")
    if not r1:
        raise ValueError("Could not find 'Region 1' in the template sheet.")
    label_col = r1[1]

    written = 0

    for region in REGIONS:
        region_row = find_row_in_column(ws, label_col, region)
        if region_row is None:
            continue

        r_slice = counts_df[counts_df["region"] == region]

        for metric in METRICS_TO_WRITE:
            metric_row = find_row_in_column(ws, label_col, metric, start_row=region_row, end_row=region_row + 200)
            if metric_row is None:
                continue

            m_slice = r_slice[r_slice["metric"] == metric]
            for _, rr in m_slice.iterrows():
                col = month_to_col_fixed(rr["month"])
                if col is None:
                    continue
                ws.cell(metric_row, col, value=int(rr["value"]))
                written += 1

    wb.save(output_path)
    print("✅ Saved:", output_path)
    print("Written cells:", written)

# -----------------------
# 1) LOAD RAW TAB2 DATA
# -----------------------
hdr = detect_header_row(input_path, sheet_name)
print("Detected header row:", hdr)

if hdr is None:
    raise ValueError("Could not detect header row with: region, metric, seq_memb_id, month. "
                     "You are likely reading the wrong sheet OR the sheet has no headers.")

df_tab2 = pd.read_excel(input_path, sheet_name=sheet_name, header=hdr)
df_tab2 = normalize_columns(df_tab2)

print("Loaded columns:", df_tab2.columns.tolist())
print("Loaded rows:", len(df_tab2))

# -----------------------
# 2) BUILD counts
# -----------------------
required = ["region","metric","seq_memb_id","month"]
missing = [c for c in required if c not in df_tab2.columns]
if missing:
    raise KeyError(f"Missing columns after load: {missing}")

df_tab2["region"] = df_tab2["region"].astype(str).str.strip()
df_tab2["metric"] = df_tab2["metric"].astype(str).str.strip()
df_tab2["month"]  = df_tab2["month"].astype(str).str.strip()

df_tab2 = df_tab2[df_tab2["region"].isin(REGIONS)]
df_tab2 = df_tab2[df_tab2["month"].isin(MONTH_ORDER)]

counts = (
    df_tab2.groupby(["region","metric","month"])["seq_memb_id"]
           .nunique()
           .reset_index(name="value")
)

print("Counts rows:", len(counts))
print("Counts metrics:", sorted(counts["metric"].unique()))

# Force template metric names in counts
metric_map = {
    "Initial HRA Enrollees #": "Initial HRA Enrollees #",
    "Annual HRA Enrollees #": "Annual HRA Enrollees #",
    "Initial HRA Completion #": "Initial HRA Completion #",
    "Annual HRA Completed #": "Annual HRA Completed #",
}
counts["metric"] = counts["metric"].astype(str).str.strip().replace(metric_map)

print("Metrics AFTER mapping:", sorted(counts["metric"].unique()))

# -----------------------
# 3) WRITE TO TEMPLATE
# -----------------------
fill_tab2_fixed_month_columns(template_path, output_path, counts, sheet_name=sheet_name)


Detected header row: 8
Loaded columns: ['region', 'metric', 'seq_memb_id', 'month']
Loaded rows: 15192
Counts rows: 384
Counts metrics: ['Annual HRA Completed #', 'Annual HRA Enrollees #', 'Initial HRA Completion #', 'Initial HRA Enrollees #']
Metrics AFTER mapping: ['Annual HRA Completed #', 'Annual HRA Enrollees #', 'Initial HRA Completion #', 'Initial HRA Enrollees #']
✅ Saved: /Users/reinaldoburgos/sources/My_Space/my_projects/PHM01_Test/output_files/PHM01_Report_filled_20251231_091725.xlsx
Written cells: 384


In [8]:
# show what openpyxl sees near Region 1 in the label column
wb = load_workbook(template_path)
ws = wb["Health Assessment(HRA)"]
ws3= wb["Health Assessment(HRA)"]
r1 = find_first_match_anywhere(ws, "Region 1")
label_col = r1[1]
for r in range(r1[0], r1[0] + 20):
    v = ws.cell(r, label_col).value
    if v:
        print(r, repr(v))


19 'Region 1'
20 'Initial HRA Enrollees #'
21 'Initial HRA Completion #'
22 'HRA Percentage %'
23 'Annual HRA Enrollees #'
24 'Annual HRA Completed #'
25 'Annual HRA Completed %'
27 'Region 2'
28 'Initial HRA Enrollees #'
29 'Initial HRA Completion #'
30 'HRA Percentage %'
31 'Annual HRA Enrollees #'
32 'Annual HRA Completed #'
33 'Annual HRA Completed %'
35 'Region 3'
36 'Initial HRA Enrollees #'
37 'Initial HRA Completion #'
38 'HRA Percentage %'


### Tab 3

In [9]:
# -----------------------
# CONFIG (edit if needed)
# -----------------------
TAB3_TEMPLATE_SHEET = "Health Assessment(HRA)"
TAB3_START_ROW = 12

REGIONS = [f"Region {i}" for i in range(1, 9)]
MONTH_ORDER = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
MONTH_START_COL = 3   # C
MONTH_END_COL = 14    # N

IDENTITY_COLS = [
    "Gender at Birth",
    "Identify as",
    "Sexual Orientation",
    "Race",
    "Ethnicity",
]

# -----------------------
# BUILD Tab3 counts (creates Sub_Category dynamically)
# -----------------------
def build_tab3_counts(tab3_raw: pd.DataFrame, distinct_members=True) -> pd.DataFrame:
    df = tab3_raw.rename(columns={
        "Region": "region",
        "Seq_Memb_Id": "seq_memb_id",
        "Month": "month",
    }).copy()

    required = ["region","seq_memb_id","month"] + IDENTITY_COLS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns for Tab3: {missing}\nFound: {df.columns.tolist()}")

    df["region"] = df["region"].astype(str).str.strip()
    df["month"]  = df["month"].astype(str).str.strip()

    df = df[df["region"].isin(REGIONS) & df["month"].isin(MONTH_ORDER)]

    long_df = df.melt(
        id_vars=["region","month","seq_memb_id"],
        value_vars=IDENTITY_COLS,
        var_name="category",
        value_name="sub_category"
    )

    long_df["sub_category"] = long_df["sub_category"].astype(str).str.strip()
    long_df = long_df[
        long_df["sub_category"].notna()
        & (long_df["sub_category"] != "")
        & (long_df["sub_category"].str.lower() != "nan")
    ]

    if distinct_members:
        out = (long_df.groupby(["region","category","sub_category","month"])["seq_memb_id"]
                      .nunique()
                      .reset_index(name="value"))
    else:
        out = (long_df.groupby(["region","category","sub_category","month"])
                      .size()
                      .reset_index(name="value"))

    out["region"] = pd.Categorical(out["region"], categories=REGIONS, ordered=True)
    out["month"]  = pd.Categorical(out["month"], categories=MONTH_ORDER, ordered=True)
    out = out.sort_values(["region","category","sub_category","month"]).reset_index(drop=True)
    return out

# -----------------------
# MERGE-SAFE cell writer
# -----------------------
def _find_merge_top_left(ws, row, col):
    for rng in ws.merged_cells.ranges:
        if (row, col) in rng:
            return rng.min_row, rng.min_col
    return row, col

def safe_set(ws, row, col, value):
    cell = ws.cell(row=row, column=col)
    if isinstance(cell, MergedCell):
        r0, c0 = _find_merge_top_left(ws, row, col)
        ws.cell(r0, c0).value = value
    else:
        cell.value = value

def month_to_col_fixed(m):
    m = str(m).strip()
    if m not in MONTH_ORDER:
        return None
    return MONTH_START_COL + MONTH_ORDER.index(m)

def unmerge_overlapping(ws, min_row, min_col, max_row, max_col):
    # copy list because we will modify merges while iterating
    merges = list(ws.merged_cells.ranges)
    for rng in merges:
        if rng.max_row < min_row or rng.min_row > max_row:
            continue
        if rng.max_col < min_col or rng.min_col > max_col:
            continue
        ws.unmerge_cells(str(rng))

# -----------------------
# WRITE Tab3 into template
# -----------------------
def fill_tab3_template(template_path, output_path, tab3_counts: pd.DataFrame,
                       sheet_name=TAB3_TEMPLATE_SHEET, start_row=TAB3_START_ROW):
    wb = load_workbook(template_path)
    if sheet_name not in wb.sheetnames:
        raise KeyError(f"Template sheet '{sheet_name}' not found. Sheets: {wb.sheetnames}")
    ws = wb[sheet_name]

    # Clear existing merges in the area we will rewrite (A..N from start_row down)
    # This prevents "MergedCell read-only" errors from old template merges.
    unmerge_overlapping(ws, min_row=start_row, min_col=1, max_row=ws.max_row, max_col=MONTH_END_COL)

    r = start_row

    # For fast lookup: (region, category, sub_category, month) -> value
    # (so we can fill zeros then overwrite)
    key_map = {}
    for _, rr in tab3_counts.iterrows():
        key_map[(str(rr["region"]), str(rr["category"]), str(rr["sub_category"]), str(rr["month"]))] = int(rr["value"])

    for region in REGIONS:
        region_rows_start = r

        region_df = tab3_counts[tab3_counts["region"] == region]
        if region_df.empty:
            continue

        # region label (we will merge later)
        safe_set(ws, r, 1, region)

        for category in IDENTITY_COLS:
            cat_df = region_df[region_df["category"] == category]
            if cat_df.empty:
                continue

            cat_rows_start = r
            safe_set(ws, r, 2, category)

            subcats = sorted(cat_df["sub_category"].dropna().unique().tolist())

            for sc in subcats:
                # Subcategory label
                safe_set(ws, r, 3, sc)

                # Fill ALL months with 0 first
                for col in range(MONTH_START_COL, MONTH_END_COL + 1):
                    safe_set(ws, r, col, 0)

                # Overwrite months that exist with actual values
                for m in MONTH_ORDER:
                    v = key_map.get((region, category, sc, m))
                    if v is not None:
                        col = month_to_col_fixed(m)
                        safe_set(ws, r, col, v)

                r += 1

            # Merge category over its subcategories
            if r - 1 > cat_rows_start:
                ws.merge_cells(start_row=cat_rows_start, start_column=2, end_row=r-1, end_column=2)

            # blank row between categories
            r += 1

        # Merge region over everything written for that region
        region_rows_end = r - 1
        if region_rows_end > region_rows_start:
            ws.merge_cells(start_row=region_rows_start, start_column=1, end_row=region_rows_end, end_column=1)

        # blank row between regions
        r += 1

    wb.save(output_path)
    print("✅ Saved:", output_path)

# -----------------------
# RUN (example)
# -----------------------
# tab3_raw = pd.read_excel(input_path, sheet_name=TAB3_INPUT_SHEET, header=YOUR_HEADER_ROW)
# tab3_counts = build_tab3_counts(tab3_raw, distinct_members=True)  # or False
# fill_tab3_template(template_path, output_path, tab3_counts)



In [ ]:
# tab3_raw = pd.read_excel(input_path, sheet_name="HRA Population Metrics", skiprows=8, 
#                         usecols=('Region','Gender at Birth', 'Identify as', 'Sexual Orientation', 'Race', 'Ethnicity' ,'Seq_Memb_Id','Month'))
# -------------------------
# SHARED SETTINGS (same idea as Tab2)
# -------------------------
REGIONS = [f"Region {i}" for i in range(1, 9)]
MONTH_ORDER = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

# Template month values go into C..N
MONTH_START_COL = 3  # C

# Tab3 template layout
TAB3_SHEET = "HRA Population"   # <-- change if your template uses a different name
START_ROW = 12                 # <-- where Region 1 section begins (adjust once if needed)
REGION_COL = 1                 # A
CATEGORY_COL = 2               # B
SUBCAT_COL = 3                 # C

# These are the identity columns in your raw data
IDENTITY_COLS = [
    "Gender at Birth",
    "Identify as",
    "Sexual Orientation",
    "Race",
    "Ethnicity",
]


# -------------------------
# HELPERS
# -------------------------
def month_to_col_fixed(month: str):
    """Map Jan..Dec to C..N"""
    m = str(month).strip()
    if m not in MONTH_ORDER:
        return None
    return MONTH_START_COL + MONTH_ORDER.index(m)

def normalize_tab3_raw(df: pd.DataFrame) -> pd.DataFrame:
    """Rename your raw columns to standard names used downstream."""
    df = tab3_raw.copy()
    df = df.rename(columns={
        "Region": "region",
        "Seq_Memb_Id": "seq_memb_id",
        "Month": "month",
    })
    # keep identity columns as-is (they include spaces)
    return df

def build_tab3_counts(tab3_raw: pd.DataFrame, distinct_members: bool = True) -> pd.DataFrame:
    """
    Returns long aggregated table with columns:
      region, category, sub_category, month, value
    """
    df = normalize_tab3_raw(tab3_raw)

    required = ["region", "seq_memb_id", "month"] + IDENTITY_COLS
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns for Tab3: {missing}\nFound: {df.columns.tolist()}")

    df["region"] = df["region"].astype(str).str.strip()
    df["month"]  = df["month"].astype(str).str.strip()

    df = df[df["region"].isin(REGIONS) & df["month"].isin(MONTH_ORDER)]

    # Wide -> long: Category/Subcategory
    long_df = df.melt(
        id_vars=["region", "month", "seq_memb_id"],
        value_vars=IDENTITY_COLS,
        var_name="category",
        value_name="sub_category",
    )

    # Clean subcategories
    long_df["sub_category"] = long_df["sub_category"].astype(str).str.strip()
    long_df = long_df[
        long_df["sub_category"].notna()
        & (long_df["sub_category"] != "")
        & (long_df["sub_category"].str.lower() != "nan")
    ]

    # Aggregate
    if distinct_members:
        out = (long_df.groupby(["region","category","sub_category","month"])["seq_memb_id"]
                      .nunique()
                      .reset_index(name="value"))
    else:
        out = (long_df.groupby(["region","category","sub_category","month"])
                      .size()
                      .reset_index(name="value"))

    # Stable ordering
    out["region"] = pd.Categorical(out["region"], categories=REGIONS, ordered=True)
    out["month"]  = pd.Categorical(out["month"], categories=MONTH_ORDER, ordered=True)
    out = out.sort_values(["region","category","sub_category","month"]).reset_index(drop=True)
    return out

def fill_tab3_identity(template_path, output_path, tab3_counts: pd.DataFrame,
                       sheet_name=TAB3_SHEET, start_row=START_ROW):
    """
    Writes Tab3 dynamically:
      Col A: Region (merged down per region block)
      Col B: Category (merged down per category block)
      Col C: Subcategory (one row per subcategory)
      Col C..N: month values
      Blank row after each category and each region
    """
    wb = load_workbook(template_path)
    if sheet_name not in wb.sheetnames:
        raise KeyError(f"Sheet '{sheet_name}' not found. Sheets: {wb.sheetnames}")
    ws = wb[sheet_name]

    d = tab3_counts.copy()
    d["region"] = d["region"].astype(str)
    d["category"] = d["category"].astype(str)
    d["sub_category"] = d["sub_category"].astype(str)
    d["month"] = d["month"].astype(str)

    r = start_row

    for region in REGIONS:
        region_df = d[d["region"] == region]
        if region_df.empty:
            continue

        region_start = r
        ws.cell(row=r, column=REGION_COL, value=region)

        # Keep categories in your defined order (and still allow missing ones)
        for category in IDENTITY_COLS:
            cat_df = region_df[region_df["category"] == category]
            if cat_df.empty:
                continue

            cat_start = r
            ws.cell(row=r, column=CATEGORY_COL, value=category)

            subcats = sorted(cat_df["sub_category"].dropna().unique().tolist())

            for sc in subcats:
                sc_df = cat_df[cat_df["sub_category"] == sc]

                ws.cell(row=r, column=SUBCAT_COL, value=sc)

                # write month values
                for _, rr in sc_df.iterrows():
                    col = month_to_col_fixed(rr["month"])
                    if col is not None:
                        ws.cell(row=r, column=col, value=int(rr["value"]))

                r += 1

            # merge Category label down
            if r - 1 > cat_start:
                ws.merge_cells(start_row=cat_start, start_column=CATEGORY_COL,
                               end_row=r-1, end_column=CATEGORY_COL)

            # blank row after category block
            r += 1

        # merge Region label down the full block
        region_end = r - 1
        if region_end > region_start:
            ws.merge_cells(start_row=region_start, start_column=REGION_COL,
                           end_row=region_end, end_column=REGION_COL)

        # blank row after region block
        r += 1

    wb.save(output_path)
    print("✅ Saved:", output_path)

In [ ]:
tab3_counts = build_tab3_counts(tab3_raw, distinct=True)   # or distinct=False
fill_tab3_identity(
    template_path=template_path,
    output_path=output_path,
    tab3_raw=tab3_counts,
    sheet_name="HRA Population Metrics",
    start_row=12
)

In [ ]:
print(tab3_raw.columns.tolist())

In [ ]:
tab3_raw = pd.read_excel(input_path, sheet_name="HRA Population Metrics", skiprows=8, 
                        usecols=('Region','Gender at Birth', 'Identify as', 'Sexual Orientation', 'Race', 'Ethnicity' ,'Seq_Memb_Id','Month'))
tab3_raw.columns = tab3_raw.columns.str.strip()

IDENTITY_COLS = {
    "Gender at Birth": "gender_at_birth",
    "Identified as": "identified_as",
    "Sexual Orientation": "sexual_orientation",
    "Race": "race",
    "Ethnicity": "ethnicity",
}

MONTH_START_COL = 3  # C
MONTH_ORDER = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
REGIONS = [f"Region {i}" for i in range(1, 9)]
CATEGORIES_ORDER = ["Gender at Birth", "Identified as", "Sexual Orientation", "Race", "Ethnicity"]

def month_to_col_fixed(month):
    if month not in MONTH_ORDER:
        return None
    return MONTH_START_COL + MONTH_ORDER.index(month)

def fill_tab3_identity(
    template_path,
    output_path,
    tab3_raw,                       # columns: region, category, sub_category, month, value
    sheet_name="HRA Population",   # <-- your Tab3 sheet name
    start_row=12,                  # <-- where the first Region block starts (adjust once)
    region_col=1, category_col=2, subcat_col=3
):
    wb = load_workbook(template_path)
    ws = wb[sheet_name]

    # Ensure ordering
    d = tab3_raw.copy()
    d["region"] = d["Region"].astype(str).str.strip()
    d["category"] = d["category"].astype(str).str.strip()
    d["sub_category"] = d["sub_category"].astype(str).str.strip()
    d["month"] = d["month"].astype(str).str.strip()

    # sort to keep output stable
    d["region"] = pd.Categorical(d["Region"], categories=REGIONS, ordered=True)
    d["category"] = pd.Categorical(d["category"], categories=CATEGORIES_ORDER, ordered=True)
    d = d.sort_values(["region","category","sub_category","month"]).reset_index(drop=True)

    r = start_row

    # helper: write a whole row of months for a given slice
    def write_month_values(row_idx, slice_df):
        for _, rr in slice_df.iterrows():
            col = month_to_col_fixed(rr["month"])
            if col:
                ws.cell(row=row_idx, column=col, value=int(rr["value"]))

    for region in REGIONS:
        region_df = d[d["region"] == region]
        if region_df.empty:
            continue

        region_start = r

        # Region label goes in col A at region_start (merged later)
        ws.cell(row=r, column=region_col, value=region)

        for category in CATEGORIES_ORDER:
            cat_df = region_df[region_df["category"] == category]
            if cat_df.empty:
                continue

            cat_start = r
            # Category label in col B (merged later)
            ws.cell(row=r, column=category_col, value=category)

            # Subcategories dynamic list
            subcats = cat_df["sub_category"].dropna().astype(str).unique().tolist()
            subcats.sort()

            for sc in subcats:
                sc_df = cat_df[cat_df["sub_category"] == sc]

                ws.cell(row=r, column=subcat_col, value=sc)
                write_month_values(r, sc_df)
                r += 1

            # merge the Category label down its subcats
            if r - 1 > cat_start:
                ws.merge_cells(start_row=cat_start, start_column=category_col,
                               end_row=r-1, end_column=category_col)

            # blank row after each category block
            r += 1

        # merge region label down the whole region block (exclude trailing blank if you want)
        region_end = r - 1
        if region_end > region_start:
            ws.merge_cells(start_row=region_start, start_column=region_col,
                           end_row=region_end, end_column=region_col)

        # extra blank row after each region block
        r += 1

    wb.save(output_path)
    print("✅ Saved:", output_path)

In [ ]:
    print(tab3_raw.columns)

In [ ]:
# tab3_counts = build_tab3_counts(tab3_raw, distinct=True)   # or distinct=False
# fill_tab3_identity(
#     template_path=template_path,
#     output_path=output_path,
#     tab3_raw=tab3_counts,
#     sheet_name="HRA Population Metrics",
#     start_row=12
# )

# -------------------------
# RUN TAB3 (you provide tab3_raw + paths)
# -------------------------
# Example:
tab3_raw = pd.read_excel(input_path, sheet_name="HRA Population Metrics")  # or however you already load it
tab3_counts = build_tab3_counts(tab3_raw, distinct_members=True)  # set False for total rows
fill_tab3_identity(template_path, output_path, tab3_counts, sheet_name="HRA Population Metrics", start_row=8)


In [ ]:


ID_COL = "Seq_Memb_Id"
REGION_COL = "Region"
MONTH_COL = "Month"   # <-- change if yours is different
MONTH_ORDER = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
ALLOWED = {
    "Gender at Birth": {
        "Male", "Female", "Other", "Unknown", "Prefer not to say"
    },
    "Identify as": {
        "Agender", "Non-binary", "Transgender", "Cisgender", "Genderqueer", "Other", "Unknown", "Prefer not to say"
    },
    "Sexual Orientation": {
        "Straight", "Gay", "Lesbian", "Bisexual", "Queer", "Questioning", "Other", "Unknown", "Prefer not to say"
    },
    "Race": {
        "White", "Black or African American", "African Americans",
        "American Indian or Alaska Native", "Alaska Natives",
        "Asian", "Native Hawaiian or Other Pacific Islander",
        "Other", "Unknown", "Prefer not to say"
    },
    "Ethnicity": {
        "Hispanic or Latino", "Not Hispanic or Latino",
        "Hispanic", "Latino", "Non-Hispanic",
        "Other", "Unknown", "Prefer not to say"
    }
}

long_df["Month"] = (
    long_df[MONTH_COL]
      .astype("string")
      .str.strip()
)

CAT_COLS = [
    "Gender at Birth",
    "Identify as",
    "Sexual Orientation",
    "Race",
    "Ethnicity",
]

REGIONS = [f"Region {i}" for i in range(1, 9)]

long_df = (
    tab3_raw[tab3_raw[REGION_COL].isin(REGIONS)][[ID_COL, REGION_COL, MONTH_COL] + CAT_COLS]
      .melt(
          id_vars=[ID_COL, REGION_COL, MONTH_COL],
          value_vars=CAT_COLS,
          var_name="Category",
          value_name="Sub_Category",
      )
)

long_df["Sub_Category"] = long_df["Sub_Category"].astype("string").str.strip()
long_df = long_df.dropna(subset=["Sub_Category"])
long_df = long_df[long_df["Sub_Category"].ne("")]

# Month already Jan/Feb/... → no datetime parsing
long_df["Month"] = long_df[MONTH_COL].astype("string").str.strip()

result = (
    long_df
    .groupby([REGION_COL, "Category", "Sub_Category", "Month"])[ID_COL]
    .nunique()
    .reset_index(name="Distinct_Seq_Memb_id")
)
print(result.columns)

In [ ]:


result["Month"] = pd.Categorical(
    result["Month"],
    categories=MONTH_ORDER,
    ordered=True
)

result = result.sort_values(
    ["Region", "Category", "Sub_Category", "Month"]
)


result[REGION_COL] = pd.Categorical(result[REGION_COL], categories=REGIONS, ordered=True)
result = result.sort_values([REGION_COL, "Category", "Sub_Category", "Month"])

# clean values first
long_df["Sub_Category"] = (
    long_df["Sub_Category"].astype("string").str.strip()
)

# Keep only values that are valid for that Category
mask = long_df.apply(
    lambda r: r["Sub_Category"] in ALLOWED.get(r["Category"], set()),
    axis=1
)

long_df_valid = long_df[mask].copy()


In [ ]:
pivot_months = result.pivot_table(
    index=["Region", "Category", "Sub_Category"],
    columns="Month",
    values="Distinct_Seq_Memb_id",
    fill_value=0,
    observed=True,   # <-- keep all months & regions
    aggfunc="sum"
).reset_index()

month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
cols_in_pivot = [c for c in month_order if c in pivot_months.columns]
pivot_months = pivot_months[[REGION_COL, "Category", "Sub_Category"] + cols_in_pivot]

tab3_raw.columns[tab3_raw.columns.str.contains("Race|Ethnicity|Sexual|Identify|Gender", case=False, na=False)]
tab3_raw.columns = tab3_raw.columns.astype(str).str.strip()

pivot_months

In [ ]:
print(tab2_raw.columns.tolist())
print(tab3_raw.columns.tolist())

In [ ]:
def _load_wb(template_path: str, output_path: str):
    """Open output if it exists; otherwise start from template."""
    path = output_path if os.path.exists(output_path) else template_path
    wb = load_workbook(path)
    return wb

def _save_wb(wb, output_path: str):
    wb.save(output_path)

def _normalize(s):
    if s is None:
        return ""
    return str(s).strip()

def _month_col_map_from_header(ws, header_row: int, start_col: int = 1, end_col: int = 200):
    """Map month label -> column index by scanning a header row."""
    m = {}
    for c in range(start_col, end_col + 1):
        v = _normalize(ws.cell(header_row, c).value)
        if v:
            m[v] = c
    return m

def _safe_write(ws, row: int, col: int, value):
    """
    If (row,col) is inside a merged range, write to the merged range's top-left.
    Otherwise write to (row,col).
    """
    target_row, target_col = row, col
    for rng in ws.merged_cells.ranges:
        if rng.min_row <= row <= rng.max_row and rng.min_col <= col <= rng.max_col:
            target_row, target_col = rng.min_row, rng.min_col
            break
    ws.cell(target_row, target_col).value = value

def _unique_count(df: pd.DataFrame, seq_col: str) -> int:
    if df.empty:
        return 0
    return df[seq_col].nunique(dropna=True)

In [ ]:
def _parse_bucket(bucket: str):
    """
    Parse template bucket strings like '0-17', '18-24', '25-34', '65+'.
    Returns (lo, hi) where hi can be None for '+'.
    """
    b = _normalize(bucket)
    m = re.match(r"^\s*(\d+)\s*-\s*(\d+)\s*$", b)
    if m:
        return int(m.group(1)), int(m.group(2))
    m = re.match(r"^\s*(\d+)\s*\+\s*$", b)
    if m:
        return int(m.group(1)), None
    return None

def _age_to_bucket(age_value, available_buckets):
    """
    If age_value is already a bucket string like '25-34', return it if present.
    If it's numeric, map it to one of the available buckets from the sheet.
    """
    # Already bucket-like
    if isinstance(age_value, str):
        v = _normalize(age_value)
        if v in available_buckets:
            return v
        # if user gave '25 - 34' with spaces, normalize attempt
        v2 = re.sub(r"\s+", "", v)
        for b in available_buckets:
            if re.sub(r"\s+", "", _normalize(b)) == v2:
                return b
        return None

    # Numeric
    try:
        a = int(age_value)
    except Exception:
        return None

    # Build parsed buckets
    parsed = []
    for b in available_buckets:
        p = _parse_bucket(b)
        if p:
            parsed.append((b, p[0], p[1]))

    # Prefer exact range match
    for b, lo, hi in parsed:
        if hi is None:
            if a >= lo:
                return b
        else:
            if lo <= a <= hi:
                return b
    return None

#### Tab 2

In [ ]:
# ---------------- Tab 2 ----------------

def fill_tab2_region_hra_counts_single_col(
    template_path,
    output_path,
    sheet_name,
    df_rows,
    region_col,
    metric_col,
    month_col,
    seq_col,
    month_header_row,      # row where months are written (A blank, months start at B)
    first_detail_row,      # first row where "Region 1" starts
    last_detail_row,       # last row to scan (end of Tab2 block)
    month_start_col=2,     # B=2
):
    wb = _load_wb(template_path, output_path)
    ws = wb[sheet_name]

    # Build month->column mapping from the header row (starting at column B)
    month_to_col = _month_col_map_from_header(ws, month_header_row, start_col=month_start_col, end_col=200)

    # Clean input data
    df = df_rows.copy()
    df[region_col] = df[region_col].astype(str).str.strip()
    df[metric_col] = df[metric_col].astype(str).str.strip()
    df[month_col]  = df[month_col].astype(str).str.strip()

    # Pre-aggregate unique members per (region, metric, month)
    grouped = (
        df.groupby([region_col, metric_col, month_col], dropna=False)[seq_col]
          .nunique()
          .reset_index(name="count")
    )
    counts = {
        (r, met, mon): int(cnt)
        for r, met, mon, cnt in grouped[[region_col, metric_col, month_col, "count"]].itertuples(index=False, name=None)
    }

    current_region = None
    region_pat = re.compile(r"^\s*Region\s+\d+\s*$", re.IGNORECASE)

    # Scan the template rows
    for r in range(first_detail_row, last_detail_row + 1):
        label = _normalize(ws.cell(r, 1).value)  # Column A

        if not label:
            # blank separator row - keep current_region as-is (or set to None if you prefer)
            continue

        # Region header row (e.g., "Region 1")
        if region_pat.match(label):
            current_region = label  # keep exact text from template
            continue

        # Metric row (must have a current region)
        if not current_region:
            continue

        metric = label

        # Write all months for this (region, metric)
        months_in_input = [
            c for c in df.columns
            if c not in (input_region_col, input_metric_col)
        ]
        for mon, c in month_to_col.items():
            if mon not in months_in_input:
                continue
            v = month_vals.get(mon, 0)
            if isinstance(v, float) and pd.isna(v):
                v = 0
            _safe_write(ws, r, c, v)

    _save_wb(wb, output_path)


    input_data = pd.read_excel(input_path, sheet_name="Health Assessment(HRA)", skiprows=9)
    print(input_data.columns.tolist())  # you already confirmed these
    
    fill_tab2_from_wide_input(
        template_path=template_path,
        output_path=output_path,
        sheet_name="Health Assessment(HRA)",
        df_input=input_data,
        input_region_col="Region",
        input_metric_col="Metric",
        month_header_row=11,    # <-- set to your template's month header row
        first_detail_row=12,    # <-- row where Region 1 starts
        last_detail_row=200,   # <-- adjust to end of block
        month_start_col=2,     # months start at column B
        skip_metric_regex=r"%" # skip % rows
    )


In [ ]:
def find_month_header_row(template_path, sheet_name, search_rows=200):
    wb = load_workbook(template_path, data_only=False)
    ws = wb[sheet_name]
    for r in range(1, search_rows + 1):
        row_vals = [str(ws.cell(r, c).value).strip() if ws.cell(r, c).value is not None else "" for c in range(1, 40)]
        if "Jan" in row_vals and "Feb" in row_vals and "Mar" in row_vals:
            print("Found month header row:", r)
            print("Row preview:", row_vals[:20])
            return r
    print("No month header row found in first", search_rows, "rows")
    return None

#hdr = find_month_header_row(template_path, "Health Assessment(HRA)", search_rows=200)

    input_data = pd.read_excel(input_path, sheet_name="Health Assessment(HRA)", skiprows=9)
    fill_tab2_from_wide_input_debug(
        template_path=template_path,
        output_path=output_path,
        sheet_name="Health Assessment(HRA)",
        df_input=input_data,
        month_header_row=11,     # <-- adjust if needed
        first_detail_row=12,     # <-- adjust if needed
        last_detail_row=200
    )
print("Saved to:", output_path)

def preview_col_a(template_path, sheet_name, start_row, end_row):
    wb = load_workbook(template_path, data_only=False)
    ws = wb[sheet_name]
    for r in range(start_row, end_row + 1):
        v = ws.cell(r, 1).value
        if v is not None and str(v).strip():
            print(r, repr(str(v).strip()))

# Example: if month header is on row hdr, start scanning right below it:
    preview_col_a(template_path, "Health Assessment(HRA)", start_row=hdr+1, end_row=hdr+60)

    input_data = pd.read_excel(input_path, sheet_name="Health Assessment(HRA)", skiprows=9)
    fill_tab2_from_wide_input_debug(
        template_path=template_path,
        output_path=output_path,
        sheet_name="Health Assessment(HRA)",
        df_input=df_hra,
        month_header_row=11,     # <-- adjust if needed
        first_detail_row=12,     # <-- adjust if needed
        last_detail_row=200
    )
print("Saved to:", output_path)


### Tab 3

In [ ]:
# ---------------- Tab 3 ----------------

def fill_tab3_region_identity(template_path, output_path, sheet_name, df_rows,
                              region_col, category_col, subcat_col, month_col, seq_col,
                              start_col, first_detail_row, last_detail_row):
    wb = _load_wb(template_path, output_path)
    ws = wb[sheet_name]

    df = df_rows.copy()
    df[month_col] = df[month_col].astype(str).str.strip()
    df[region_col] = df[region_col].astype(str).str.strip()
    df[category_col] = df[category_col].astype(str).str.strip()
    df[subcat_col] = df[subcat_col].astype(str).str.strip()

    for r in range(first_detail_row, last_detail_row + 1):
        reg = _normalize(ws.cell(r, 1).value)
        cat = _normalize(ws.cell(r, 2).value)
        sub = _normalize(ws.cell(r, 3).value)

        if not reg or not cat or not sub:
            continue

        # months horizontally from start_col
        for i, m in enumerate(MONTHS):
            c = start_col + i
            subdf = df[(df[region_col] == reg) & (df[category_col].str.lower() == cat.lower()) &
                       (df[subcat_col] == sub) & (df[month_col] == m)]
            _safe_write(ws, r, c, _unique_count(subdf, seq_col))

    _save_wb(wb, output_path)

### Tab 4

In [ ]:
# ---------------- Tab 4 ----------------

def fill_tab4_identity_age(template_path, output_path, sheet_name, df_rows,
                           age_col, category_col, subcat_col, month_col, seq_col,
                           header_row, first_detail_row, last_detail_row):
    wb = _load_wb(template_path, output_path)
    ws = wb[sheet_name]

    month_to_col = _month_col_map_from_header(ws, header_row, start_col=1, end_col=200)

    # Collect available age buckets from the sheet in the detail range
    age_buckets = []
    for r in range(first_detail_row, last_detail_row + 1):
        v = _normalize(ws.cell(r, 1).value)
        if v:
            age_buckets.append(v)
    age_buckets = sorted(set(age_buckets), key=lambda x: (len(x), x))

    df = df_rows.copy()
    df[month_col] = df[month_col].astype(str).str.strip()
    df[category_col] = df[category_col].astype(str).str.strip()
    df[subcat_col] = df[subcat_col].astype(str).str.strip()

    # Map df age -> template bucket
    df["_age_bucket_"] = df[age_col].apply(lambda x: _age_to_bucket(x, age_buckets))
    df = df[df["_age_bucket_"].notna()].copy()

    for r in range(first_detail_row, last_detail_row + 1):
        age_bucket = _normalize(ws.cell(r, 1).value)
        cat = _normalize(ws.cell(r, 2).value)
        sub = _normalize(ws.cell(r, 3).value)

        if not age_bucket or not cat or not sub:
            continue

        for m, c in month_to_col.items():
            if m not in MONTHS:
                continue
            subdf = df[(df["_age_bucket_"] == age_bucket) &
                       (df[category_col].str.lower() == cat.lower()) &
                       (df[subcat_col] == sub) &
                       (df[month_col] == m)]
            _safe_write(ws, r, c, _unique_count(subdf, seq_col))

    _save_wb(wb, output_path)

### Tab 5

In [ ]:
# ---------------- Tab 5 ----------------

def fill_tab5_region_needs(template_path, output_path, sheet_name, df_rows,
                           region_col, need_col, month_col, seq_col,
                           start_col, first_detail_row, last_detail_row):
    wb = _load_wb(template_path, output_path)
    ws = wb[sheet_name]

    df = df_rows.copy()
    df[month_col] = df[month_col].astype(str).str.strip()
    df[region_col] = df[region_col].astype(str).str.strip()
    df[need_col] = df[need_col].astype(str).str.strip()

    for r in range(first_detail_row, last_detail_row + 1):
        reg = _normalize(ws.cell(r, 1).value)
        need = _normalize(ws.cell(r, 2).value)

        if not reg or not need:
            continue

        for i, m in enumerate(MONTHS):
            c = start_col + i
            subdf = df[(df[region_col] == reg) & (df[need_col] == need) & (df[month_col] == m)]
            _safe_write(ws, r, c, _unique_count(subdf, seq_col))

    _save_wb(wb, output_path)

### Tab 6

In [ ]:
# ---------------- Tab 6 ----------------

def fill_tab6_month_totals_simple(template_path, output_path, sheet_name, df_rows,
                                  month_col, seq_col, jan_col, target_row, scope):
    wb = _load_wb(template_path, output_path)
    ws = wb[sheet_name]

    df = df_rows.copy()
    df[month_col] = df[month_col].astype(str).str.strip()

    if str(scope).lower() == "quarter":
        months = MONTHS[:3]  # Jan/Feb/Mar
    else:
        months = MONTHS[:]   # all year

    for i, m in enumerate(months):
        c = jan_col + i
        subdf = df[df[month_col] == m]
        _safe_write(ws, target_row, c, _unique_count(subdf, seq_col))

    _save_wb(wb, output_path)